# Notebook 01 — Data Preprocessing and Model Training

This notebook covers the first phase of the project:
- Loading and exploring the Adult Income dataset
- Cleaning the data (missing values, duplicates, formatting)
- Saving sensitive columns (sex, race, age) before encoding — needed for subgroup analysis
- Encoding categorical variables and preparing features
- Training a Random Forest classifier
- Saving all outputs to Google Drive for use in notebooks 02 and 03

## Dataset: Adult Income (UCI Repository)
Predict whether a person earns more than $50K/year based on census data.
48,842 instances — 14 features — binary target (>50K vs <=50K)

## Step 1 — Install and Import Libraries
We install the ucimlrepo package to download the dataset directly from UCI.
All other libraries (pandas, numpy, scikit-learn) are already available on Colab.

In [1]:
!pip install ucimlrepo -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import pickle
import os
from google.colab import drive

In [3]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/XAIP/data/'
os.makedirs(BASE, exist_ok=True)

print(f"✓ Drive mounted")
print(f"✓ Output folder ready: {BASE}")

Mounted at /content/drive
✓ Drive mounted
✓ Output folder ready: /content/drive/MyDrive/XAIP/data/


In [4]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

print(f"Feature matrix shape: {X_raw.shape}")
print(f"Target shape:         {y_raw.shape}")
print(f"\nFeature names:\n{X_raw.columns.tolist()}")
print(f"\nData types:\n{X_raw.dtypes}")

Feature matrix shape: (48842, 14)
Target shape:         (48842, 1)

Feature names:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Data types:
age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
dtype: object


## Step 4 — Initial Data Exploration
Before cleaning, we inspect the dataset to understand:
- The distribution of the target variable
- Which columns have missing values (encoded as '?' in this dataset)
- The distribution of sensitive attributes (sex, race, age)
  that will be used later for subgroup analysis

Note: the Adult Income dataset encodes missing values as '?' strings,
not as NaN — we need to handle this explicitly during cleaning.

In [5]:
df = X_raw.copy()
df['income'] = y_raw.values.ravel()

print(f"Total rows: {len(df)}")

print(f"\n=== Target distribution ===")
print(df['income'].value_counts())
print("Note: '<=50K' and '<=50K.' are the same class (dot is a formatting artifact)")

print(f"\n=== Missing values (encoded as '?') ===")
df_check = df.replace('?', np.nan)
missing = df_check.isnull().sum()
print(missing[missing > 0])

print(f"\n=== Sensitive attributes distribution ===")
print("\nSex:")
print(df['sex'].str.strip().value_counts())
print("\nRace:")
print(df['race'].str.strip().value_counts())
print("\nAge (min / max / mean):")
print(f"  min={df['age'].min()},  max={df['age'].max()},  mean={df['age'].mean():.1f}")

Total rows: 48842

=== Target distribution ===
income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64
Note: '<=50K' and '<=50K.' are the same class (dot is a formatting artifact)

=== Missing values (encoded as '?') ===
workclass         2799
occupation        2809
native-country     857
dtype: int64

=== Sensitive attributes distribution ===

Sex:
sex
Male      32650
Female    16192
Name: count, dtype: int64

Race:
race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64

Age (min / max / mean):
  min=17,  max=90,  mean=38.6


## Step 5 — Data Cleaning
We apply the following cleaning steps in order:

1. Strip whitespace from all string columns
   (the dataset has trailing spaces in many categorical values)
2. Replace '?' with NaN
   (UCI Adult uses '?' as the missing value indicator)
3. Remove duplicate rows
4. Drop rows with any NaN value
5. Reset the index

After cleaning we verify the final shape and report how many rows were removed.

In [6]:
df_clean = df.copy()

# 1. Strip whitespace from string columns
str_cols = df_clean.select_dtypes(include='object').columns
df_clean[str_cols] = df_clean[str_cols].apply(lambda col: col.str.strip())

# 2. Replace '?' with NaN
df_clean.replace('?', np.nan, inplace=True)

# 3. Remove duplicates
n_before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f"Duplicate rows removed: {n_before - len(df_clean)}")

# 4. Drop rows with NaN
n_before = len(df_clean)
df_clean.dropna(inplace=True)
print(f"Rows with NaN removed:  {n_before - len(df_clean)}")

# 5. Reset index
df_clean.reset_index(drop=True, inplace=True)

print(f"\nOriginal shape: {df.shape}")
print(f"Clean shape:    {df_clean.shape}")
print(f"Total removed:  {len(df) - len(df_clean)} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")

Duplicate rows removed: 29
Rows with NaN removed:  3619

Original shape: (48842, 15)
Clean shape:    (45194, 15)
Total removed:  3648 rows (7.5%)


## Step 6 — Save Sensitive Columns Before Encoding
This is a critical step for our project.

We save the original readable values of sex, race, and age BEFORE applying
label encoding, because after encoding we lose the original labels
(e.g. 'Male'/'Female' become 0/1).

We also create age groups (bins) to define meaningful demographic subgroups:
  <25  |  25-35  |  35-50  |  50-65  |  65+

These sensitive columns will be attached to the test set and used in
notebook 03 to compute subgroup-level faithfulness metrics.

In [7]:
sensitive = df_clean[['sex', 'race', 'age']].copy()

# Create age group bins for subgroup analysis
sensitive['age_group'] = pd.cut(
    sensitive['age'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=['<25', '25-35', '35-50', '50-65', '65+']
)

print("=== Sensitive attributes after cleaning ===")
print("\nSex:")
print(sensitive['sex'].value_counts())
print("\nAge groups:")
print(sensitive['age_group'].value_counts().sort_index())
print("\nRace:")
print(sensitive['race'].value_counts())

# Safety check: must be aligned with df_clean
assert len(sensitive) == len(df_clean), "ERROR: sensitive/df_clean misalignment"
print("\n✓ Alignment check passed")

=== Sensitive attributes after cleaning ===

Sex:
sex
Male      30509
Female    14685
Name: count, dtype: int64

Age groups:
age_group
<25       8428
25-35    12069
35-50    16017
50-65     7337
65+       1343
Name: count, dtype: int64

Race:
race
White                 38877
Black                  4227
Asian-Pac-Islander     1302
Amer-Indian-Eskimo      435
Other                   353
Name: count, dtype: int64

✓ Alignment check passed


## Step 7 — Encoding
We encode the data to make it suitable for the Random Forest classifier:

1. Target encoding: '>50K' and '>50K.' → 1  |  '<=50K' and '<=50K.' → 0
   We use str.contains('>50K') to handle both variants with and without the dot.

2. Label encoding for all categorical features:
   Each unique string value is mapped to an integer (e.g. Male→1, Female→0).
   We store the encoders in le_dict in case we need to reverse the mapping later.

In [8]:
df_enc = df_clean.copy()

# 1. Binary target encoding
df_enc['income'] = (df_enc['income'].str.contains('>50K')).astype(int)

print("=== Target distribution after encoding ===")
print(df_enc['income'].value_counts())
print(f"Positive class (>50K): {df_enc['income'].mean():.2%}")

# 2. Label encoding for categorical columns
cat_cols = df_enc.select_dtypes(include='object').columns.tolist()
print(f"\nCategorical columns to encode: {cat_cols}")

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    le_dict[col] = le
    print(f"  ✓ {col}: {le.classes_.tolist()}")

=== Target distribution after encoding ===
income
0    33988
1    11206
Name: count, dtype: int64
Positive class (>50K): 24.80%

Categorical columns to encode: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
  ✓ workclass: ['Federal-gov', 'Local-gov', 'Private', 'Self-emp-inc', 'Self-emp-not-inc', 'State-gov', 'Without-pay']
  ✓ education: ['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th', '9th', 'Assoc-acdm', 'Assoc-voc', 'Bachelors', 'Doctorate', 'HS-grad', 'Masters', 'Preschool', 'Prof-school', 'Some-college']
  ✓ marital-status: ['Divorced', 'Married-AF-spouse', 'Married-civ-spouse', 'Married-spouse-absent', 'Never-married', 'Separated', 'Widowed']
  ✓ occupation: ['Adm-clerical', 'Armed-Forces', 'Craft-repair', 'Exec-managerial', 'Farming-fishing', 'Handlers-cleaners', 'Machine-op-inspct', 'Other-service', 'Priv-house-serv', 'Prof-specialty', 'Protective-serv', 'Sales', 'Tech-support', 'Transport-moving']
  ✓ relat

In [9]:
#-------------------------------- Train test split (training 80% test 20%) ----------------------------------------

X = df_enc.drop(columns=['income'])
y = df_enc['income']

feature_names = X.columns.tolist()
print(f"Features ({len(feature_names)}): {feature_names}")

# Single split call — sensitive aligned automatically
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Reset all indices
X_train    = X_train.reset_index(drop=True)
X_test     = X_test.reset_index(drop=True)
y_train    = y_train.reset_index(drop=True)
y_test     = y_test.reset_index(drop=True)
sens_train = sens_train.reset_index(drop=True)
sens_test  = sens_test.reset_index(drop=True)

print(f"\nTrain set: {X_train.shape}  |  positive: {y_train.mean():.2%}")
print(f"Test set:  {X_test.shape}  |  positive: {y_test.mean():.2%}")

# Alignment check
assert len(X_test) == len(sens_test), "ERROR: X_test / sens_test misalignment"
print("✓ Alignment check passed")

Features (14): ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Train set: (36155, 14)  |  positive: 24.80%
Test set:  (9039, 14)  |  positive: 24.79%
✓ Alignment check passed


## Step 9 — Model Training
We train a Random Forest classifier on the training set.

Random Forest was chosen because:
- It is a non-linear black-box model — exactly the type of model
  for which post-hoc explanations (SHAP, LIME) are necessary
- It performs well on tabular data without extensive hyperparameter tuning
- TreeExplainer (used in notebook 02 for SHAP) is optimized for tree-based models

Hyperparameters:
- n_estimators=100 → 100 decision trees
- random_state=42  → reproducibility
- n_jobs=-1        → use all available CPU cores (speeds up training)

In [10]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("=== Model Performance on Test Set ===")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

=== Model Performance on Test Set ===
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.90      6798
        >50K       0.74      0.62      0.67      2241

    accuracy                           0.85      9039
   macro avg       0.81      0.77      0.79      9039
weighted avg       0.85      0.85      0.85      9039

Accuracy: 0.8520


In [11]:
# ----------------------- SAVE TO DRIVE --------------------------

# 1. Training set
X_train_save = X_train.copy()
X_train_save['income'] = y_train.values
X_train_save.to_csv(BASE + 'adult_train.csv', index=False)

# 2. Test set with sensitive columns attached
X_test_save = X_test.copy()
X_test_save['income']    = y_test.values
X_test_save['sex_raw']   = sens_test['sex'].values
X_test_save['race_raw']  = sens_test['race'].values
X_test_save['age_group'] = sens_test['age_group'].values
X_test_save.to_csv(BASE + 'adult_test.csv', index=False)

# 3. Trained model
with open(BASE + 'model_rf.pkl', 'wb') as f:
    pickle.dump(model, f)

# 4. Feature names
with open(BASE + 'feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

# Verify saved files
print("✓ Files saved to Drive:")
for fname in sorted(os.listdir(BASE)):
    size_kb = os.path.getsize(BASE + fname) / 1024
    print(f"   {fname:30s}  {size_kb:.1f} KB")



✓ Files saved to Drive:
   adult_test.csv                  506.4 KB
   adult_train.csv                 1398.8 KB
   feature_names.pkl               0.2 KB
   model_rf.pkl                    84724.2 KB


In [12]:
print("=" * 60)
print("  SUMMARY — OUTPUTS FOR COLLEAGUES")
print("=" * 60)
print(f"\nBASE PATH:\n  {BASE}")
print(f"\nFILES PRODUCED:")
print(f"  adult_train.csv   → {len(X_train)} rows, {len(feature_names)} features")
print(f"  adult_test.csv    → {len(X_test)} rows + sex_raw, race_raw, age_group")
print(f"  model_rf.pkl      → Random Forest (100 trees, accuracy={accuracy_score(y_test,y_pred):.3f})")
print(f"  feature_names.pkl → {len(feature_names)} features")
print(f"\nFEATURE NAMES:")
for i, f in enumerate(feature_names):
    print(f"  {i:2d}. {f}")
print(f"\nSUBGROUPS IN TEST SET:")
print(f"  sex_raw:   {sens_test['sex'].unique().tolist()}")
print(f"  race_raw:  {sens_test['race'].unique().tolist()}")
print(f"  age_group: {sorted(sens_test['age_group'].dropna().unique().tolist())}")
print(f"\n✓ Preprocessing complete — ready for notebooks 02 and 03")

  SUMMARY — OUTPUTS FOR COLLEAGUES

BASE PATH:
  /content/drive/MyDrive/XAIP/data/

FILES PRODUCED:
  adult_train.csv   → 36155 rows, 14 features
  adult_test.csv    → 9039 rows + sex_raw, race_raw, age_group
  model_rf.pkl      → Random Forest (100 trees, accuracy=0.852)
  feature_names.pkl → 14 features

FEATURE NAMES:
   0. age
   1. workclass
   2. fnlwgt
   3. education
   4. education-num
   5. marital-status
   6. occupation
   7. relationship
   8. race
   9. sex
  10. capital-gain
  11. capital-loss
  12. hours-per-week
  13. native-country

SUBGROUPS IN TEST SET:
  sex_raw:   ['Male', 'Female']
  race_raw:  ['White', 'Black', 'Asian-Pac-Islander', 'Other', 'Amer-Indian-Eskimo']
  age_group: ['25-35', '35-50', '50-65', '65+', '<25']

✓ Preprocessing complete — ready for notebooks 02 and 03
